# vn-longsum — Phase 2: fine-tune ViT5 + LoRA trên Colab (T4)

Notebook này **chỉ gọi script trong repo**; mọi hyperparameter nằm ở `configs/train.yaml`.

1. Runtime → Change runtime type → **T4 GPU**.
2. Sửa `REPO_URL` ở cell cấu hình.
3. Run all. Nếu Colab ngắt kết nối: chạy lại từ đầu — train tự **resume** từ checkpoint mới nhất trên Drive.

In [ ]:
!nvidia-smi

## Cấu hình đường dẫn

In [ ]:
REPO_URL   = "https://github.com/<username>/vn-longsum.git"   # <- sửa
DRIVE_ROOT = "/content/drive/MyDrive/vn-longsum"
RUN_DIR    = f"{DRIVE_ROOT}/runs/vit5-lora"                   # checkpoint + final_adapter
MERGED_DIR = f"{DRIVE_ROOT}/models/vit5-vnsum-merged"        # model đã merge cho phase 4
DATA_DRIVE = f"{DRIVE_ROOT}/data/processed"                   # cache output phase 1

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## Lấy code

In [ ]:
%cd /content
!test -d vn-longsum && (cd vn-longsum && git pull --ff-only) || git clone {REPO_URL} vn-longsum
%cd /content/vn-longsum

## Cài dependency
Không cài `torch` (dùng bản CUDA có sẵn của Colab) và không cài đè pandas/numpy/pyarrow.

In [ ]:
!pip install -q datasets==4.8.5 transformers==5.16.1 peft==0.21.2 accelerate==1.15.0 rouge-score==0.1.2
!pip install -q -e . --no-deps

## HF token (tùy chọn)
Thêm `HF_TOKEN` vào 🔑 Secrets của Colab.

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception as e:
    print("Không dùng HF_TOKEN:", e)

## Dữ liệu phase 1
Lần đầu: chạy `prepare_data.py` (~10 phút CPU) rồi lưu lên Drive. Các lần sau: copy từ Drive.

In [ ]:
!if [ -f "{DATA_DRIVE}/stats.json" ]; then mkdir -p data && cp -r "{DATA_DRIVE}" data/; else python scripts/prepare_data.py --config configs/data.yaml --log-level WARNING && mkdir -p "{DATA_DRIVE}" && cp -r data/processed/. "{DATA_DRIVE}/"; fi
!cat data/processed/stats.json | head -5

## Kiểm tra nhanh

In [ ]:
!python -m pytest -q

## Train
Chạy lại đúng cell này sau khi bị ngắt kết nối → tự resume (`resume: auto`).
Muốn thử fp16 (nhanh hơn, có rủi ro NaN với T5): thêm `--precision fp16`.

In [ ]:
!python -m vnsum.models.train --config configs/train.yaml --output_dir "{RUN_DIR}"

## Merge LoRA → model đầy đủ cho phase 4

In [ ]:
!python scripts/merge_lora.py --config configs/train.yaml --adapter "{RUN_DIR}/final_adapter" --output "{MERGED_DIR}"